In [1]:
%%writefile warp_tile.cu
#include<iostream>
#include<cuda_runtime.h>
using namespace std;

#define BLOCK_M 64
#define BLOCK_N 64
#define BLOCK_K 16
#define WARP_M 32
#define WARP_N 32
#define THREAD_M 8
#define THREAD_N 4
#define RUNS 100

__global__ void warp_tile(float *A , float *B , float *C , int M , int N , int K){
  __shared__ float sA[BLOCK_M][BLOCK_K];
  __shared__ float sB[BLOCK_K][BLOCK_N];

  // thread id
  int tx = threadIdx.x;

  //block tile
  int row = blockIdx.y * BLOCK_M ;
  int col = blockIdx.x * BLOCK_N ;

  // warp & lane
  int warp_id = threadIdx.x / 32;
  int lane = threadIdx.x % 32;

  // warp row and  warp col
  int warp_row = warp_id / 2;
  int warp_col = warp_id % 2;

  // this is offset which tells the warp that where its tile begine in block
  int warp_row_offset = warp_row * WARP_M;
  int warp_col_offset = warp_col * WARP_N;

  //Thread mapping inside the warp
    int thread_row = lane / 8;
    int thread_col = lane % 8;

    int thread_row_offset = thread_row * THREAD_M;
    int thread_col_offset = thread_col * THREAD_N;

  float sum[THREAD_M][THREAD_N] = {0.0f};

  for(int k0= 0; k0 < K; k0+=BLOCK_K){

    for(int i = 0; i < 8; i++){
      int index = tx + i * blockDim.x;

      int load_row_A = index / BLOCK_K;
      int load_col_A = index % BLOCK_K;

      // load the A tile
      if(load_row_A < BLOCK_M && load_col_A < BLOCK_K){
        sA[load_row_A][load_col_A] = A[(row + load_row_A) * K + (k0 + load_col_A)];
      }

      // load the B tile
      int load_row_B = index / BLOCK_N;
      int load_col_B = index % BLOCK_N;

      if(load_row_B < BLOCK_K && load_col_B < BLOCK_N){
        sB[load_row_B][load_col_B] = B[(k0 + load_row_B) * N + (col + load_col_B)];
      }
    }
    __syncthreads();

    // compute
    for(int k = 0; k < BLOCK_K; k++){
      float regA[THREAD_M];
      float regB[THREAD_N];

      for(int i=0; i < THREAD_M; i++){
        regA[i] = sA[warp_row_offset + thread_row_offset + i][k];
      }
      for(int j=0; j <THREAD_N; j++){
        regB[j] = sB[k][warp_col_offset + thread_col_offset + j];
      }

      for(int i = 0; i < THREAD_M; i++){
        for(int j= 0; j < THREAD_N; j++){
          sum[i][j] += regA[i] * regB[j];
        }
      }

    }
    __syncthreads();
  }

  //Write result back to the global memory
  for(int i=0; i < THREAD_M; i++){
    for(int j = 0; j < THREAD_N; j++){
      int global_row = row + warp_row_offset + thread_row_offset + i;
      int global_col = col + warp_col_offset + thread_col_offset + j;

      if(global_row < M && global_col < N){
        C[global_row * N + global_col] = sum[i][j];
      }
    }
  }
}

#define CUDA_CHECK(call) \
{ \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        cout << "CUDA error: " << cudaGetErrorString(err) \
             << " at " << __FILE__ << ":" << __LINE__ << endl; \
        exit(1); \
    } \
}

// Host code
int main(){
  int M = 1024;
  int N = 1024;
  int K = 1024;

  int sizeA = M * K * sizeof(float);
  int sizeB = K * N * sizeof(float);
  int sizeC = M * N * sizeof(float);

  //Host memory allocation
  float *h_A = new float[M*K];
  float *h_B = new float[K*N];
  float *h_C = new float[M*N];

  // Initializing
  for(int i = 0; i < M*K; i++) h_A[i] = 4.0f;
  for(int i = 0; i < K*N; i++) h_B[i] = 4.0f;

  // Device memory allocation
  float *d_A , *d_B , *d_C;
  CUDA_CHECK(cudaMalloc((void**)&d_A , sizeA));
  CUDA_CHECK(cudaMalloc((void**)&d_B , sizeB));
  CUDA_CHECK(cudaMalloc((void**)&d_C , sizeC));

  //copy data from host to device
  CUDA_CHECK(cudaMemcpy(d_A , h_A , sizeA , cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_B , h_B , sizeB , cudaMemcpyHostToDevice));

  dim3 block(128);

  dim3 grid(
    (N + BLOCK_N - 1) / BLOCK_N,
    (M + BLOCK_M - 1) / BLOCK_M
  );

  //kernel launch
  warp_tile<<<grid , block >>>(d_A , d_B, d_C , M , N , K);
  cudaDeviceSynchronize();

  cudaEvent_t start , stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);

  cudaEventRecord(start);

  for(int i=0; i<RUNS; i++){
    warp_tile<<<grid , block>>>(d_A, d_B, d_C, M , N , K);
  }

  cudaEventRecord(stop);
  cudaEventSynchronize(stop);

  float ms;

  cudaEventElapsedTime(&ms, start, stop);
  ms /= RUNS;

  double flops = 2.0 * M * N * K;
  double gflops = flops / (ms / 1000.0) / 1.e9;

  cout<<"Averge Kernel Time:" << ms << "ms\n";
  cout<<"GFLOPS :" << gflops << endl;

  //copy data back to host from device
  CUDA_CHECK(cudaMemcpy(h_C , d_C , sizeC , cudaMemcpyDeviceToHost));

  cout<<"Sample Result :" << h_C[0] << endl;

  //free device memory
  cudaFree(d_A);
  cudaFree(d_B);
  cudaFree(d_C);

  // free host memory
  delete[] h_A;
  delete[] h_B;
  delete[] h_C;

  return 0;

}



Writing warp_tile.cu


In [2]:
!nvcc warp_tile.cu -o warp_tile -arch=sm_75;

In [3]:
!./warp_tile

Averge Kernel Time:1.86037ms
GFLOPS :1154.33
Sample Result :16384
